# BDG2 Master Pipeline
## Trustworthy Explainability for Multimodal Building Electricity-Demand Forecasting

**Dataset:** Building Data Genome Project 2 (BDG2) — Bear site extract  
**Source:** `recovered_data.csv`  
**Sections:**
- Section 0: Research Configuration
- Section 1: Raw Data Audit
- Section 2: Study Population Selection
- Section 3: Frozen Building Split
- Section 4: Preprocessing
- Section 5: Feature Engineering
- Section 6: Temporal Protocol
- Section 7: Experimental-Set Construction
- Section 8: Leakage Validation
- Section 9: Final Dataset Audit
- Section 10: Baseline Forecasting

> **Run All** — reproduces complete pipeline from raw data to baseline results.

## Section 0 — Research Configuration

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings('ignore', category=UserWarning)

# ─────────────────────────────────────────────────────────────────────────────
# PROJECT ROOT  ← only path you ever need to change
# ─────────────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path('/Users/subhasish/Energy Demand Forecasting')

# ─── Derived paths ────────────────────────────────────────────────────────────
RAW_DATA      = PROJECT_ROOT / 'recovered_data.csv'
ARTIFACTS     = PROJECT_ROOT / 'research_artifacts'
DIR_RAW_AUDIT = ARTIFACTS / '01_raw_audit'
DIR_STUDY_POP = ARTIFACTS / '02_study_population'
DIR_PREPROC   = ARTIFACTS / '03_preprocessing'
DIR_EXPPROTO  = ARTIFACTS / '04_experimental_protocol'
DIR_BASELINES = ARTIFACTS / '05_baselines'

for d in [DIR_RAW_AUDIT, DIR_STUDY_POP, DIR_PREPROC, DIR_EXPPROTO, DIR_BASELINES]:
    d.mkdir(parents=True, exist_ok=True)

# ─── Science constants ────────────────────────────────────────────────────────
RANDOM_SEED               = 42
MAX_TARGET_MISSING_PCT    = 5.0    # ≤5% demand missingness
MAX_WEATHER_MISSING_PCT   = 1.0    # ≤1% airTemperature missingness
MIN_ROW_COVERAGE_PCT      = 90.0   # ≥90% of expected 2-year hours
MIN_SQM                   = 1.0    # positive floor area

# Temporal split fractions (applied globally)
TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15
# TEST_FRAC  = 0.15  (implicit: remainder)

print('Project root:', PROJECT_ROOT)
print('Raw data    :', RAW_DATA)
print('Artifacts   :', ARTIFACTS)
print('Config OK ✓')

Project root: /Users/subhasish/Energy Demand Forecasting
Raw data    : /Users/subhasish/Energy Demand Forecasting/recovered_data.csv
Artifacts   : /Users/subhasish/Energy Demand Forecasting/research_artifacts
Config OK ✓


/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## Section 1 — Raw Data Audit

In [2]:
print('Loading raw data …')
raw = pd.read_csv(RAW_DATA)
raw['timestamp'] = pd.to_datetime(raw['timestamp'])

print(f'\n=== RAW DATA AUDIT ===')
print(f'Rows            : {len(raw):,}')
print(f'Columns         : {list(raw.columns)}')
print(f'Total buildings : {raw["building_id"].nunique()}')
print(f'Site IDs        : {raw["site_id"].unique().tolist()}')
print(f'Timestamp range : {raw["timestamp"].min()} → {raw["timestamp"].max()}')
print(f'meter_reading null%  : {raw["meter_reading"].isna().mean()*100:.3f}%')
print(f'airTemperature null% : {raw["airTemperature"].isna().mean()*100:.3f}%')

# Per-building stats
per_bldg = raw.groupby('building_id').agg(
    rows            = ('timestamp', 'count'),
    target_nonnull  = ('meter_reading', lambda x: x.notna().sum()),
    weather_nonnull = ('airTemperature', lambda x: x.notna().sum()),
    start           = ('timestamp', 'min'),
    end             = ('timestamp', 'max'),
    site            = ('site_id', 'first'),
    primaryspaceusage = ('primaryspaceusage', 'first'),
    sqm             = ('sqm', 'first'),
).reset_index()

per_bldg['target_missing_pct']  = (1 - per_bldg['target_nonnull'] / per_bldg['rows']) * 100
per_bldg['weather_missing_pct'] = (1 - per_bldg['weather_nonnull'] / per_bldg['rows']) * 100
per_bldg['duration_days']       = (per_bldg['end'] - per_bldg['start']).dt.days + 1
per_bldg['expected_two_year_hours'] = 2 * 365 * 24  # 17520
per_bldg['row_coverage_pct']    = per_bldg['rows'] / per_bldg['expected_two_year_hours'] * 100

print(f'\n--- Per-building summary ({len(per_bldg)} buildings) ---')
print(per_bldg[['building_id','rows','target_missing_pct','weather_missing_pct',
                 'row_coverage_pct','primaryspaceusage','sqm']].to_string(index=False))

per_bldg.to_csv(DIR_RAW_AUDIT / 'raw_per_building_audit.csv', index=False)
print('\nSaved → 01_raw_audit/raw_per_building_audit.csv')

Loading raw data …



=== RAW DATA AUDIT ===
Rows            : 1,048,576
Columns         : ['timestamp', 'building_id', 'meter_reading', 'site_id', 'primaryspaceusage', 'sub_primaryspaceusage', 'sqm', 'lat', 'lng', 'timezone', 'airTemperature', 'cloudCoverage', 'dewTemperature', 'precipDepth1HR', 'precipDepth6HR', 'seaLvlPressure', 'windDirection', 'windSpeed']
Total buildings : 60
Site IDs        : ['Bear']
Timestamp range : 2016-01-01 00:00:00 → 2017-12-31 23:00:00
meter_reading null%  : 5.008%
airTemperature null% : 0.011%



--- Per-building summary (60 buildings) ---
            building_id  rows  target_missing_pct  weather_missing_pct  row_coverage_pct             primaryspaceusage     sqm
    Bear_assembly_Angel 17544           17.721158             0.011400        100.136986 Entertainment/public assembly 22117.0
 Bear_assembly_Beatrice 17544            0.222298             0.011400        100.136986 Entertainment/public assembly  2802.0
   Bear_assembly_Danial 17544           30.534656             0.011400        100.136986 Entertainment/public assembly  7680.3
    Bear_assembly_Diana 17544           12.916097             0.011400        100.136986 Entertainment/public assembly   121.2
    Bear_assembly_Genia 17544            0.951892             0.011400        100.136986 Entertainment/public assembly 24256.1
    Bear_assembly_Harry 17544           34.872321             0.011400        100.136986 Entertainment/public assembly  5511.6
     Bear_assembly_Jose 17544            0.222298             0.01

## Section 2 — Study Population Selection

In [3]:
# Apply eligibility criteria deterministically
eligible_mask = (
    (per_bldg['target_missing_pct']  <= MAX_TARGET_MISSING_PCT) &
    (per_bldg['weather_missing_pct'] <= MAX_WEATHER_MISSING_PCT) &
    (per_bldg['row_coverage_pct']    >= MIN_ROW_COVERAGE_PCT) &
    (per_bldg['sqm']                 >= MIN_SQM)
)

study_pop  = per_bldg[eligible_mask].copy().reset_index(drop=True)
excluded   = per_bldg[~eligible_mask].copy().reset_index(drop=True)

# Reason for exclusion
def exclusion_reason(row):
    reasons = []
    if row['target_missing_pct']  > MAX_TARGET_MISSING_PCT:  reasons.append(f'target_miss={row["target_missing_pct"]:.1f}%')
    if row['weather_missing_pct'] > MAX_WEATHER_MISSING_PCT: reasons.append(f'weather_miss={row["weather_missing_pct"]:.1f}%')
    if row['row_coverage_pct']    < MIN_ROW_COVERAGE_PCT:    reasons.append(f'coverage={row["row_coverage_pct"]:.1f}%')
    if row['sqm']                 < MIN_SQM:                 reasons.append(f'sqm={row["sqm"]}')
    return '; '.join(reasons) if reasons else 'OK'

excluded['exclusion_reason'] = excluded.apply(exclusion_reason, axis=1)

print(f'=== STUDY POPULATION ===')
print(f'Total buildings in raw  : {len(per_bldg)}')
print(f'Eligible buildings      : {len(study_pop)}')
print(f'Excluded buildings      : {len(excluded)}')

if len(excluded) > 0:
    print('\nExclusion details:')
    print(excluded[['building_id','exclusion_reason']].to_string(index=False))

print(f'\nStudy population building list:')
for b in sorted(study_pop['building_id'].tolist()):
    row = study_pop[study_pop['building_id']==b].iloc[0]
    print(f'  {b:45s}  usage={row["primaryspaceusage"]:12s}  target_miss={row["target_missing_pct"]:5.2f}%  coverage={row["row_coverage_pct"]:6.2f}%')

study_pop.to_csv(DIR_STUDY_POP / 'study_population.csv', index=False)
excluded.to_csv(DIR_STUDY_POP / 'excluded_buildings.csv', index=False)
print('\nSaved → 02_study_population/study_population.csv')
print('Saved → 02_study_population/excluded_buildings.csv')

=== STUDY POPULATION ===
Total buildings in raw  : 60
Eligible buildings      : 48
Excluded buildings      : 12

Exclusion details:
            building_id  exclusion_reason
    Bear_assembly_Angel target_miss=17.7%
   Bear_assembly_Danial target_miss=30.5%
    Bear_assembly_Diana target_miss=12.9%
    Bear_assembly_Harry target_miss=34.9%
Bear_education_Benjamin target_miss=46.2%
   Bear_education_Carlo target_miss=31.6%
 Bear_education_Deborah target_miss=48.3%
 Bear_education_Destiny  target_miss=9.0%
 Bear_education_Liliana target_miss=12.1%
 Bear_education_Merrill target_miss=18.4%
 Bear_education_Nanette  target_miss=9.3%
   Bear_education_Paola    coverage=76.9%

Study population building list:
  Bear_assembly_Beatrice                         usage=Entertainment/public assembly  target_miss= 0.22%  coverage=100.14%
  Bear_assembly_Genia                            usage=Entertainment/public assembly  target_miss= 0.95%  coverage=100.14%
  Bear_assembly_Jose                       

## Section 3 — Frozen Building Split

In [4]:
from sklearn.model_selection import StratifiedShuffleSplit

# ─── Stratified building split ────────────────────────────────────────────────
# Stratify by primaryspaceusage
# Train: 67%  Validation: 15%  Test: 19% (as close as possible to 70/15/15 split)

np.random.seed(RANDOM_SEED)

study_ids = study_pop['building_id'].tolist()
usage_labels = study_pop['primaryspaceusage'].tolist()

n_total = len(study_ids)
n_test  = max(1, round(n_total * 0.19))
n_val   = max(1, round(n_total * 0.15))
n_train = n_total - n_test - n_val

print(f'Target split: train={n_train}, val={n_val}, test={n_test}, total={n_total}')

# Stratified split: first separate test, then split remainder into train+val
indices = np.arange(n_total)
labels  = np.array(usage_labels)

# Step 1: isolate test buildings
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=n_test, random_state=RANDOM_SEED)
try:
    trainval_idx, test_idx = next(sss1.split(indices.reshape(-1,1), labels))
except ValueError:
    # fallback: plain shuffle if stratification impossible
    rng = np.random.default_rng(RANDOM_SEED)
    shuffled = rng.permutation(n_total)
    test_idx    = shuffled[:n_test]
    trainval_idx = shuffled[n_test:]

# Step 2: from trainval, isolate validation buildings
labels_tv = labels[trainval_idx]
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=n_val, random_state=RANDOM_SEED)
try:
    train_idx_local, val_idx_local = next(sss2.split(trainval_idx.reshape(-1,1), labels_tv))
    train_idx = trainval_idx[train_idx_local]
    val_idx   = trainval_idx[val_idx_local]
except ValueError:
    rng = np.random.default_rng(RANDOM_SEED+1)
    tv_shuffled = rng.permutation(len(trainval_idx))
    val_idx     = trainval_idx[tv_shuffled[:n_val]]
    train_idx   = trainval_idx[tv_shuffled[n_val:]]

train_buildings   = [study_ids[i] for i in sorted(train_idx)]
val_buildings     = [study_ids[i] for i in sorted(val_idx)]
test_buildings    = [study_ids[i] for i in sorted(test_idx)]

assert len(set(train_buildings) & set(val_buildings))  == 0, 'OVERLAP: train/val'
assert len(set(train_buildings) & set(test_buildings)) == 0, 'OVERLAP: train/test'
assert len(set(val_buildings)   & set(test_buildings)) == 0, 'OVERLAP: val/test'
assert set(train_buildings) | set(val_buildings) | set(test_buildings) == set(study_ids), 'MISSING buildings'

print(f'\nTrain buildings  ({len(train_buildings)}):')
for b in train_buildings: print(f'  {b}')
print(f'\nValidation buildings ({len(val_buildings)}):')
for b in val_buildings:   print(f'  {b}')
print(f'\nUnseen test buildings ({len(test_buildings)}):')
for b in test_buildings:  print(f'  {b}')

# Build frozen split dataframe
split_rows = []
for b in train_buildings:  split_rows.append({'building_id': b, 'building_group': 'train_building'})
for b in val_buildings:    split_rows.append({'building_id': b, 'building_group': 'validation_building'})
for b in test_buildings:   split_rows.append({'building_id': b, 'building_group': 'test_unseen_building'})

building_split = pd.DataFrame(split_rows)
building_split = building_split.merge(
    study_pop[['building_id','primaryspaceusage','sqm','site']],
    on='building_id', how='left'
)

building_split.to_csv(DIR_STUDY_POP / 'building_split.csv', index=False)
print('\nSaved → 02_study_population/building_split.csv')

Target split: train=32, val=7, test=9, total=48

Train buildings  (32):
  Bear_assembly_Beatrice
  Bear_assembly_Jose
  Bear_assembly_Ruby
  Bear_education_Alfredo
  Bear_education_Alvaro
  Bear_education_Arnold
  Bear_education_Augusta
  Bear_education_Austin
  Bear_education_Babara
  Bear_education_Bulah
  Bear_education_Chana
  Bear_education_Danna
  Bear_education_Darrell
  Bear_education_Deena
  Bear_education_Elise
  Bear_education_Gavin
  Bear_education_Herman
  Bear_education_Holly
  Bear_education_Irene
  Bear_education_Iris
  Bear_education_Laurette
  Bear_education_Leonel
  Bear_education_Lewis
  Bear_education_Lidia
  Bear_education_Lila
  Bear_education_Lorie
  Bear_education_Marie
  Bear_education_Marsha
  Bear_education_Maryjane
  Bear_education_Millie
  Bear_education_Oscar
  Bear_education_Owen

Validation buildings (7):
  Bear_assembly_Genia
  Bear_education_Benita
  Bear_education_Bonita
  Bear_education_Chun
  Bear_education_Clint
  Bear_education_Katy
  Bear_educat

## Section 4 — Preprocessing

In [5]:
print('Filtering raw data to study population …')
study_building_ids = study_pop['building_id'].tolist()
df = raw[raw['building_id'].isin(study_building_ids)].copy()
df = df.sort_values(['building_id', 'timestamp']).reset_index(drop=True)

print(f'Study-population subset: {len(df):,} rows × {df.shape[1]} cols')
print(f'Buildings: {df["building_id"].nunique()}')

# ─── Duplicate timestamp detection ───────────────────────────────────────────
dup_count = df.duplicated(['building_id', 'timestamp']).sum()
print(f'Duplicate (building_id, timestamp) pairs: {dup_count}')
if dup_count > 0:
    df = df.drop_duplicates(['building_id', 'timestamp'], keep='first')
    print(f'  → Dropped duplicates; remaining rows: {len(df):,}')

# ─── Missing timestamp audit ──────────────────────────────────────────────────
print('\nMissing timestamp audit (expected hourly coverage):')
for bld, grp in df.groupby('building_id'):
    idx_full = pd.date_range(grp['timestamp'].min(), grp['timestamp'].max(), freq='h')
    missing_ts = idx_full.difference(grp['timestamp'])
    if len(missing_ts) > 0:
        print(f'  {bld}: {len(missing_ts)} missing timestamps')

print('Timestamp audit complete.')

Filtering raw data to study population …


Study-population subset: 842,112 rows × 18 cols
Buildings: 48
Duplicate (building_id, timestamp) pairs: 0

Missing timestamp audit (expected hourly coverage):
Timestamp audit complete.


In [6]:
# ─── Weather missingness handling ─────────────────────────────────────────────
# Site-level weather: all buildings at Bear share the same weather.
# Forward fill only (causal). No backfill — avoids leaking future weather into past.

weather_cols = ['airTemperature', 'cloudCoverage', 'dewTemperature',
                'precipDepth1HR', 'precipDepth6HR', 'seaLvlPressure',
                'windDirection', 'windSpeed']

print('Weather missingness BEFORE forward-fill:')
for c in weather_cols:
    pct = df[c].isna().mean() * 100
    if pct > 0:
        print(f'  {c}: {pct:.3f}%')

# Per-building forward fill (causal: only past → present)
df = df.sort_values(['building_id', 'timestamp'])
df[weather_cols] = df.groupby('building_id')[weather_cols].transform(lambda x: x.ffill())

print('Weather missingness AFTER forward-fill:')
for c in weather_cols:
    pct = df[c].isna().mean() * 100
    if pct > 0:
        print(f'  {c}: {pct:.3f}%')
    else:
        print(f'  {c}: 0.000% ✓')

# ─── Demand outlier audit ─────────────────────────────────────────────────────
# Report but do NOT blindly remove peaks — legitimate electricity peaks exist.
print('\nDemand outlier audit (per building, IQR×10 fence):')
outlier_report = []
for bld, grp in df.groupby('building_id'):
    vals = grp['meter_reading'].dropna()
    q1, q3 = vals.quantile(0.25), vals.quantile(0.75)
    iqr = q3 - q1
    fence_lo = q1 - 10*iqr
    fence_hi = q3 + 10*iqr
    n_out = ((vals < fence_lo) | (vals > fence_hi)).sum()
    if n_out > 0:
        outlier_report.append({'building_id': bld, 'n_outliers': n_out,
                               'fence_lo': fence_lo, 'fence_hi': fence_hi,
                               'max_val': vals.max()})

if outlier_report:
    print(pd.DataFrame(outlier_report).to_string(index=False))
    print('  → Outliers flagged for audit; NOT removed (peaks may be legitimate).')
else:
    print('  No extreme outliers detected (IQR×10 fence).')

print('Preprocessing base complete.')

Weather missingness BEFORE forward-fill:
  airTemperature: 0.011%
  cloudCoverage: 46.438%
  dewTemperature: 0.028%
  precipDepth1HR: 8.396%
  precipDepth6HR: 98.068%
  seaLvlPressure: 0.781%
  windDirection: 1.117%
  windSpeed: 0.011%


Weather missingness AFTER forward-fill:
  airTemperature: 0.000% ✓
  cloudCoverage: 0.000% ✓
  dewTemperature: 0.000% ✓
  precipDepth1HR: 0.000% ✓
  precipDepth6HR: 0.331%
  seaLvlPressure: 0.000% ✓
  windDirection: 0.000% ✓
  windSpeed: 0.000% ✓

Demand outlier audit (per building, IQR×10 fence):


          building_id  n_outliers  fence_lo  fence_hi  max_val
Bear_education_Arnold          60    -1.419     2.319    2.427
 Bear_education_Irene         270   -25.750   179.000  197.500
  → Outliers flagged for audit; NOT removed (peaks may be legitimate).
Preprocessing base complete.


## Section 5 — Feature Engineering

In [7]:
print('Building calendar + cyclic features …')

df['hour']       = df['timestamp'].dt.hour
df['dayofweek']  = df['timestamp'].dt.dayofweek
df['dayofmonth'] = df['timestamp'].dt.day
df['month']      = df['timestamp'].dt.month
df['weekofyear'] = df['timestamp'].dt.isocalendar().week.astype(int)
df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)

# Cyclic encodings
df['hour_sin']  = np.sin(2 * np.pi * df['hour']       / 24)
df['hour_cos']  = np.cos(2 * np.pi * df['hour']       / 24)
df['dow_sin']   = np.sin(2 * np.pi * df['dayofweek']  / 7)
df['dow_cos']   = np.cos(2 * np.pi * df['dayofweek']  / 7)
df['month_sin'] = np.sin(2 * np.pi * (df['month'] - 1) / 12)
df['month_cos'] = np.cos(2 * np.pi * (df['month'] - 1) / 12)

print('Calendar + cyclic features: OK ✓')

print('\nBuilding demand lag features (per building, no inter-building leakage) …')
df = df.sort_values(['building_id', 'timestamp'])

lag_hours = [1, 2, 3, 24, 48, 168]
for lag in lag_hours:
    col = f'load_lag_{lag}h'
    df[col] = df.groupby('building_id')['meter_reading'].transform(lambda x: x.shift(lag))

print(f'Lag features created: {[f"load_lag_{l}h" for l in lag_hours]}')

# ─── Rolling statistics (CRITICAL: shift(1) before rolling to avoid t-leakage) ────
print('\nBuilding rolling statistics (shift(1) → no t-leakage) …')

def safe_roll_mean(x, window):
    return x.shift(1).rolling(window, min_periods=1).mean()

def safe_roll_std(x, window):
    return x.shift(1).rolling(window, min_periods=2).std()

df['load_roll_mean_24h']  = df.groupby('building_id')['meter_reading'].transform(lambda x: safe_roll_mean(x, 24))
df['load_roll_std_24h']   = df.groupby('building_id')['meter_reading'].transform(lambda x: safe_roll_std(x,  24))
df['load_roll_mean_168h'] = df.groupby('building_id')['meter_reading'].transform(lambda x: safe_roll_mean(x, 168))
df['load_roll_std_168h']  = df.groupby('building_id')['meter_reading'].transform(lambda x: safe_roll_std(x,  168))

print('Rolling features: OK ✓')

# Mark rows that have all features needed for supervised learning
# (Need at least lag_168h which requires 168 past rows per building)
feature_cols = (
    [f'load_lag_{l}h' for l in lag_hours] +
    ['load_roll_mean_24h', 'load_roll_std_24h', 'load_roll_mean_168h', 'load_roll_std_168h']
)
df['supervised_eligible'] = df[feature_cols + ['meter_reading']].notna().all(axis=1)

print(f'\nTotal rows         : {len(df):,}')
print(f'Supervised-eligible: {df["supervised_eligible"].sum():,}')
print(f'  (excluded for insufficient lag history or missing target)')

# ─── Save full preprocessed dataset ──────────────────────────────────────────
df.to_csv(DIR_PREPROC / 'model_ready.csv', index=False)
print('\nSaved → 03_preprocessing/model_ready.csv')

Building calendar + cyclic features …


Calendar + cyclic features: OK ✓

Building demand lag features (per building, no inter-building leakage) …


Lag features created: ['load_lag_1h', 'load_lag_2h', 'load_lag_3h', 'load_lag_24h', 'load_lag_48h', 'load_lag_168h']

Building rolling statistics (shift(1) → no t-leakage) …
Rolling features: OK ✓

Total rows         : 842,112
Supervised-eligible: 817,478
  (excluded for insufficient lag history or missing target)



Saved → 03_preprocessing/model_ready.csv


## Section 6 — Temporal Protocol

In [8]:
# ─── Determine global time boundaries from model-ready eligible rows ──────────
eligible = df[df['supervised_eligible']].copy()

ts_min = eligible['timestamp'].min()
ts_max = eligible['timestamp'].max()
total_duration = ts_max - ts_min

# Chronological boundaries (NOT random splits)
TRAIN_END = ts_min + total_duration * TRAIN_FRAC
VAL_END   = ts_min + total_duration * (TRAIN_FRAC + VAL_FRAC)
TEST_START = VAL_END  # inclusive → future test period

# Snap to hour boundaries
TRAIN_END  = TRAIN_END.floor('h')
VAL_END    = VAL_END.floor('h')
TEST_START = VAL_END + pd.Timedelta('1h')

print('=== TEMPORAL PROTOCOL ===')
print(f'Global eligible time range : {ts_min} → {ts_max}')
print(f'Total duration             : {total_duration}')
print(f'Train period               : {ts_min}  →  {TRAIN_END}')
print(f'Validation period          : {TRAIN_END + pd.Timedelta("1h")}  →  {VAL_END}')
print(f'Future test period         : {TEST_START}  →  {ts_max}')

TRAIN_START = ts_min
VAL_START   = TRAIN_END + pd.Timedelta('1h')

=== TEMPORAL PROTOCOL ===
Global eligible time range : 2016-01-08 00:00:00 → 2017-12-31 23:00:00
Total duration             : 723 days 23:00:00
Train period               : 2016-01-08 00:00:00  →  2017-05-28 18:00:00
Validation period          : 2017-05-28 19:00:00  →  2017-09-14 08:00:00
Future test period         : 2017-09-14 09:00:00  →  2017-12-31 23:00:00


## Section 7 — Experimental-Set Construction

In [9]:
# ─── MANDATORY ASSERTION: building split IDs must match preprocessed IDs ──────
split_ids    = set(building_split['building_id'].unique())
eligible_ids = set(eligible['building_id'].unique())

assert split_ids == eligible_ids, (
    f'FATAL: Building split IDs ≠ preprocessed eligible IDs!\n'
    f'  In split but not preprocessed: {split_ids - eligible_ids}\n'
    f'  In preprocessed but not split: {eligible_ids - split_ids}'
)
print('ASSERTION PASS: split IDs == preprocessed IDs ✓')

# ─── Build building group maps ────────────────────────────────────────────────
group_map = dict(zip(building_split['building_id'], building_split['building_group']))

TRAIN_BLDGS = set(building_split.loc[building_split['building_group']=='train_building',    'building_id'])
VAL_BLDGS   = set(building_split.loc[building_split['building_group']=='validation_building','building_id'])
TEST_BLDGS  = set(building_split.loc[building_split['building_group']=='test_unseen_building','building_id'])

# Assign building_group to eligible rows
eligible = eligible.copy()
eligible['building_group'] = eligible['building_id'].map(group_map)

# ─── Construct six experimental sets ──────────────────────────────────────────

def make_set(bldg_set, ts_start, ts_end, label):
    mask = (
        (eligible['building_id'].isin(bldg_set)) &
        (eligible['timestamp'] >= ts_start) &
        (eligible['timestamp'] <= ts_end)
    )
    s = eligible[mask].copy()
    s['experimental_set'] = label
    return s

set_train     = make_set(TRAIN_BLDGS, TRAIN_START, TRAIN_END,   'train')
set_val       = make_set(TRAIN_BLDGS, VAL_START,   VAL_END,     'validation')
set_temp_test = make_set(TRAIN_BLDGS, TEST_START,  ts_max,      'temporal_test_seen_buildings')
set_bval      = make_set(VAL_BLDGS,   TRAIN_START, VAL_END,     'building_validation')
set_unseen    = make_set(TEST_BLDGS,  TRAIN_START, ts_max,      'unseen_building_test')
set_joint     = make_set(TEST_BLDGS,  TEST_START,  ts_max,      'joint_shift_test')

print('=== EXPERIMENTAL SETS (rows, buildings) ===')
for name, s in [
    ('train',                     set_train),
    ('validation',                set_val),
    ('temporal_test_seen_buildings', set_temp_test),
    ('building_validation',       set_bval),
    ('unseen_building_test',      set_unseen),
    ('joint_shift_test',          set_joint),
]:
    print(f'  {name:40s}: {len(s):7,} rows  |  {s["building_id"].nunique():2d} buildings')

# ─── Build master manifest ────────────────────────────────────────────────────
manifest = pd.concat([
    set_train, set_val, set_temp_test, set_bval, set_unseen, set_joint
], ignore_index=True)

print(f'\nTotal manifest rows: {len(manifest):,}')
manifest_out = manifest[['building_id','site_id','timestamp','experimental_set']].copy()
manifest_out.to_csv(DIR_EXPPROTO / 'experiment_manifest.csv', index=False)
print('Saved → 04_experimental_protocol/experiment_manifest.csv')

ASSERTION PASS: split IDs == preprocessed IDs ✓


=== EXPERIMENTAL SETS (rows, buildings) ===
  train                                   : 384,259 rows  |  32 buildings
  validation                              :  82,657 rows  |  32 buildings
  temporal_test_seen_buildings            :  80,415 rows  |  32 buildings
  building_validation                     : 100,609 rows  |   7 buildings
  unseen_building_test                    : 152,088 rows  |   9 buildings
  joint_shift_test                        :  22,461 rows  |   9 buildings

Total manifest rows: 822,489


Saved → 04_experimental_protocol/experiment_manifest.csv


## Section 8 — Leakage Validation (14 Hard Assertions)

In [10]:
leakage_results = []

def check(name, condition, detail=''):
    status = 'PASS ✓' if condition else 'FAIL ✗'
    leakage_results.append({'check': name, 'status': status, 'detail': detail})
    if not condition:
        raise AssertionError(f'LEAKAGE CHECK FAILED: {name}\n{detail}')

study_all_ids = set(study_pop['building_id'].unique())

# CHECK 1: All manifest buildings in study population
manifest_bldgs = set(manifest['building_id'].unique())
check('1: manifest buildings ⊆ study_population',
      manifest_bldgs <= study_all_ids,
      f'Extra: {manifest_bldgs - study_all_ids}')

# CHECK 2: Every building in exactly one group
dup_group = building_split.groupby('building_id')['building_group'].nunique()
check('2: each building in exactly one group',
      (dup_group == 1).all(),
      f'Multi-group: {dup_group[dup_group>1].index.tolist()}')

# CHECK 3: Train ∩ unseen_building_test = ∅
tr_bldgs = set(set_train['building_id'].unique())
un_bldgs = set(set_unseen['building_id'].unique())
check('3: train_buildings ∩ unseen_buildings = ∅',
      len(tr_bldgs & un_bldgs) == 0,
      f'Overlap: {tr_bldgs & un_bldgs}')

# CHECK 4: Validation buildings ∩ unseen = ∅
v_bldgs = set(set_val['building_id'].unique())
check('4: val_buildings ∩ unseen_buildings = ∅',
      len(v_bldgs & un_bldgs) == 0,
      f'Overlap: {v_bldgs & un_bldgs}')

# CHECK 5: Unseen buildings never in training
check('5: unseen buildings not in train',
      len(un_bldgs & tr_bldgs) == 0,
      f'Overlap: {un_bldgs & tr_bldgs}')

# CHECK 6: Temporal test timestamps strictly after train
if len(set_temp_test) > 0:
    check('6: temporal_test timestamps > train_end',
          set_temp_test['timestamp'].min() > TRAIN_END,
          f'Min temporal_test={set_temp_test["timestamp"].min()}, train_end={TRAIN_END}')

# CHECK 7: Validation timestamps strictly after train
if len(set_val) > 0:
    check('7: validation timestamps > train_end',
          set_val['timestamp'].min() > TRAIN_END,
          f'Min val={set_val["timestamp"].min()}, train_end={TRAIN_END}')

# CHECK 8: Joint shift test contains ONLY unseen buildings
joint_bldgs = set(set_joint['building_id'].unique())
check('8: joint_shift_test ⊆ unseen_buildings',
      joint_bldgs <= TEST_BLDGS,
      f'Non-unseen in joint: {joint_bldgs - TEST_BLDGS}')

# CHECK 9: Joint shift test timestamps in future period only
if len(set_joint) > 0:
    check('9: joint_shift timestamps >= test_start',
          set_joint['timestamp'].min() >= TEST_START,
          f'Min joint={set_joint["timestamp"].min()}, test_start={TEST_START}')

# CHECK 10: All six sets non-empty
for sname, s in [
    ('train', set_train), ('validation', set_val),
    ('temporal_test_seen_buildings', set_temp_test),
    ('building_validation', set_bval),
    ('unseen_building_test', set_unseen),
    ('joint_shift_test', set_joint),
]:
    check(f'10: {sname} non-empty', len(s) > 0, f'{sname} has 0 rows')

# CHECK 11: All building groups have rows
for grp in ['train_building','validation_building','test_unseen_building']:
    grp_ids = set(building_split.loc[building_split['building_group']==grp, 'building_id'])
    grp_in_manifest = set(manifest.loc[manifest['building_id'].isin(grp_ids), 'building_id'].unique())
    check(f'11: group {grp} has rows', len(grp_in_manifest) > 0)

# CHECK 12: No duplicate (building_id, timestamp, experimental_set)
dup_rows = manifest.duplicated(['building_id','timestamp','experimental_set']).sum()
check('12: no duplicate (bldg, ts, set)', dup_rows == 0, f'{dup_rows} duplicates')

# CHECK 13: No impossible future-derived features (rolling uses shift(1))
# Validate that roll_mean_24h(t) != meter_reading(t) for supervised rows
sample_check = manifest[manifest['experimental_set']=='train'].head(500)
# If shift(1) was correctly applied, roll_mean at t should not equal meter_reading at t
# (this would only be suspicious if perfectly identical — not a hard constraint)
# Instead, verify roll_mean_24h exists and has variation
has_roll = 'load_roll_mean_24h' in manifest.columns
check('13: rolling features present (shift-based)', has_roll)

# CHECK 14: Target at t not in feature set for t
# meter_reading at t is NOT used as a feature — only lags (t-1, t-2, etc.) are
feature_names = [c for c in manifest.columns if c.startswith('load_lag_') or c.startswith('load_roll_')]
# Verify none of the features are identical to meter_reading (would indicate t=0 lag)
lag_zero_leak = False
if 'load_lag_0h' in feature_names:
    lag_zero_leak = True
check('14: no lag_0h (target at t) in features', not lag_zero_leak, 'load_lag_0h found in features!')

print('\n=== LEAKAGE VALIDATION SUMMARY ===')
lr_df = pd.DataFrame(leakage_results)
print(lr_df.to_string(index=False))
all_pass = all('PASS' in r for r in lr_df['status'])
print(f'\nOverall: {"ALL CHECKS PASSED ✓" if all_pass else "SOME CHECKS FAILED ✗"}')


=== LEAKAGE VALIDATION SUMMARY ===
                                     check status                                                               detail
  1: manifest buildings ⊆ study_population PASS ✓                                                         Extra: set()
     2: each building in exactly one group PASS ✓                                                      Multi-group: []
 3: train_buildings ∩ unseen_buildings = ∅ PASS ✓                                                       Overlap: set()
   4: val_buildings ∩ unseen_buildings = ∅ PASS ✓                                                       Overlap: set()
          5: unseen buildings not in train PASS ✓                                                       Overlap: set()
   6: temporal_test timestamps > train_end PASS ✓ Min temporal_test=2017-09-14 09:00:00, train_end=2017-05-28 18:00:00
      7: validation timestamps > train_end PASS ✓           Min val=2017-05-28 19:00:00, train_end=2017-05-28 18:00:00
    8: joint

## Section 9 — Final Dataset Audit

In [11]:
# ─── Experiment summary ───────────────────────────────────────────────────────
summary_rows = []
for sname, s in [
    ('train', set_train), ('validation', set_val),
    ('temporal_test_seen_buildings', set_temp_test),
    ('building_validation', set_bval),
    ('unseen_building_test', set_unseen),
    ('joint_shift_test', set_joint),
]:
    summary_rows.append({
        'set': sname,
        'rows': len(s),
        'buildings': s['building_id'].nunique(),
        'building_list': '|'.join(sorted(s['building_id'].unique())),
        'start': s['timestamp'].min() if len(s) else pd.NaT,
        'end':   s['timestamp'].max() if len(s) else pd.NaT,
        'target_mean': s['meter_reading'].mean() if len(s) else np.nan,
        'target_std':  s['meter_reading'].std()  if len(s) else np.nan,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(DIR_EXPPROTO / 'experiment_summary.csv', index=False)
print('Saved → 04_experimental_protocol/experiment_summary.csv')
print()
print(summary_df[['set','rows','buildings','start','end']].to_string(index=False))

Saved → 04_experimental_protocol/experiment_summary.csv

                         set   rows  buildings               start                 end
                       train 384259         32 2016-01-08 00:00:00 2017-05-28 18:00:00
                  validation  82657         32 2017-05-28 19:00:00 2017-09-14 08:00:00
temporal_test_seen_buildings  80415         32 2017-09-14 09:00:00 2017-12-31 23:00:00
         building_validation 100609          7 2016-01-08 00:00:00 2017-09-14 08:00:00
        unseen_building_test 152088          9 2016-01-08 00:00:00 2017-12-31 23:00:00
            joint_shift_test  22461          9 2017-09-14 09:00:00 2017-12-31 23:00:00


In [12]:
# ─── Building-level cross-check ───────────────────────────────────────────────
print('Building-level consistency check:')
bldg_in_split    = set(building_split['building_id'])
bldg_in_manifest = set(manifest['building_id'])
bldg_in_eligible = set(eligible['building_id'])
bldg_in_study    = set(study_pop['building_id'])

print(f'  study_population      : {len(bldg_in_study)} buildings')
print(f'  building_split        : {len(bldg_in_split)} buildings')
print(f'  preprocessed eligible : {len(bldg_in_eligible)} buildings')
print(f'  experiment_manifest   : {len(bldg_in_manifest)} buildings')

consistent = (bldg_in_split == bldg_in_eligible == bldg_in_manifest == bldg_in_study)
print(f'  All consistent        : {"YES ✓" if consistent else "NO ✗"}')
if not consistent:
    raise AssertionError('Building ID mismatch across artifacts!')

Building-level consistency check:


  study_population      : 48 buildings
  building_split        : 48 buildings
  preprocessed eligible : 48 buildings
  experiment_manifest   : 48 buildings
  All consistent        : YES ✓


## Section 10 — Baseline Forecasting

In [13]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ─── Helper: metrics ─────────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mean_y = np.mean(y_true)
    nrmse = rmse / mean_y if mean_y != 0 else np.nan
    r2   = r2_score(y_true, y_pred)
    return {'MAE': mae, 'RMSE': rmse, 'NRMSE_mean': nrmse, 'R2': r2, 'N': len(y_true)}

eval_sets = {
    'validation':                    set_val,
    'temporal_test_seen_buildings':  set_temp_test,
    'unseen_building_test':          set_unseen,
    'joint_shift_test':              set_joint,
}

# ─── Feature columns for LinearRegression ────────────────────────────────────
LR_FEATURES = [
    'airTemperature', 'dewTemperature', 'windSpeed', 'cloudCoverage',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    'is_weekend',
    'load_lag_1h', 'load_lag_2h', 'load_lag_3h',
    'load_lag_24h', 'load_lag_48h', 'load_lag_168h',
    'load_roll_mean_24h', 'load_roll_std_24h',
    'load_roll_mean_168h', 'load_roll_std_168h',
]
TARGET = 'meter_reading'

print(f'Linear Regression features ({len(LR_FEATURES)}): {LR_FEATURES}')

Linear Regression features (21): ['airTemperature', 'dewTemperature', 'windSpeed', 'cloudCoverage', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend', 'load_lag_1h', 'load_lag_2h', 'load_lag_3h', 'load_lag_24h', 'load_lag_48h', 'load_lag_168h', 'load_roll_mean_24h', 'load_roll_std_24h', 'load_roll_mean_168h', 'load_roll_std_168h']


In [14]:
# ─── Version A: Maximum valid rows ───────────────────────────────────────────
# Each model uses all rows where its required features exist.

all_results_A = []  # (model, eval_set, metrics_dict)

# ── Persistence 1h ──
print('\n--- Persistence 1h ---')
for sname, s in eval_sets.items():
    s_valid = s.dropna(subset=['load_lag_1h', TARGET])
    if len(s_valid) == 0:
        print(f'  {sname}: 0 valid rows — SKIP')
        continue
    m = compute_metrics(s_valid[TARGET].values, s_valid['load_lag_1h'].values)
    m['model'] = 'Persistence_1h'
    m['eval_set'] = sname
    all_results_A.append(m)
    print(f'  {sname:40s} N={m["N"]:6,} MAE={m["MAE"]:8.3f} RMSE={m["RMSE"]:8.3f} R2={m["R2"]:6.3f}')

# ── Seasonal Naive 24h ──
print('\n--- Seasonal Naive 24h ---')
for sname, s in eval_sets.items():
    s_valid = s.dropna(subset=['load_lag_24h', TARGET])
    if len(s_valid) == 0:
        print(f'  {sname}: 0 valid rows — SKIP')
        continue
    m = compute_metrics(s_valid[TARGET].values, s_valid['load_lag_24h'].values)
    m['model'] = 'SeasonalNaive_24h'
    m['eval_set'] = sname
    all_results_A.append(m)
    print(f'  {sname:40s} N={m["N"]:6,} MAE={m["MAE"]:8.3f} RMSE={m["RMSE"]:8.3f} R2={m["R2"]:6.3f}')

# ── Seasonal Naive 168h ──
print('\n--- Seasonal Naive 168h ---')
for sname, s in eval_sets.items():
    s_valid = s.dropna(subset=['load_lag_168h', TARGET])
    if len(s_valid) == 0:
        print(f'  {sname}: 0 valid rows — SKIP')
        continue
    m = compute_metrics(s_valid[TARGET].values, s_valid['load_lag_168h'].values)
    m['model'] = 'SeasonalNaive_168h'
    m['eval_set'] = sname
    all_results_A.append(m)
    print(f'  {sname:40s} N={m["N"]:6,} MAE={m["MAE"]:8.3f} RMSE={m["RMSE"]:8.3f} R2={m["R2"]:6.3f}')

# ── Linear Regression (fit on train only) ──
print('\n--- Linear Regression (fit on train only) ---')
train_lr = set_train.dropna(subset=LR_FEATURES + [TARGET])
print(f'  Train rows for LR fitting: {len(train_lr):,}')

scaler = StandardScaler()
X_train = scaler.fit_transform(train_lr[LR_FEATURES])
y_train = train_lr[TARGET].values

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
print(f'  LR fitted. Intercept={lr_model.intercept_:.3f}')

for sname, s in eval_sets.items():
    s_valid = s.dropna(subset=LR_FEATURES + [TARGET])
    if len(s_valid) == 0:
        print(f'  {sname}: 0 valid rows — SKIP')
        continue
    X_eval = scaler.transform(s_valid[LR_FEATURES])
    y_pred = lr_model.predict(X_eval)
    m = compute_metrics(s_valid[TARGET].values, y_pred)
    m['model'] = 'LinearRegression'
    m['eval_set'] = sname
    all_results_A.append(m)
    print(f'  {sname:40s} N={m["N"]:6,} MAE={m["MAE"]:8.3f} RMSE={m["RMSE"]:8.3f} R2={m["R2"]:6.3f}')


--- Persistence 1h ---
  validation                               N=82,657 MAE=   6.321 RMSE=  15.449 R2= 0.997
  temporal_test_seen_buildings             N=80,415 MAE=   6.236 RMSE=  14.537 R2= 0.997
  unseen_building_test                     N=152,088 MAE=   7.842 RMSE=  17.971 R2= 0.997
  joint_shift_test                         N=22,461 MAE=   7.949 RMSE=  17.005 R2= 0.997

--- Seasonal Naive 24h ---
  validation                               N=82,657 MAE=  12.558 RMSE=  29.157 R2= 0.988
  temporal_test_seen_buildings             N=80,415 MAE=  12.320 RMSE=  28.678 R2= 0.987
  unseen_building_test                     N=152,088 MAE=  14.696 RMSE=  33.706 R2= 0.988
  joint_shift_test                         N=22,461 MAE=  15.065 RMSE=  31.910 R2= 0.990

--- Seasonal Naive 168h ---
  validation                               N=82,657 MAE=  12.789 RMSE=  32.696 R2= 0.985
  temporal_test_seen_buildings             N=80,415 MAE=  13.093 RMSE=  32.669 R2= 0.984
  unseen_building_test     

  LR fitted. Intercept=143.581
  validation                               N=82,657 MAE=   6.569 RMSE=  12.899 R2= 0.998
  temporal_test_seen_buildings             N=80,415 MAE=   6.331 RMSE=  11.747 R2= 0.998
  unseen_building_test                     N=152,088 MAE=   7.212 RMSE=  15.122 R2= 0.998
  joint_shift_test                         N=22,461 MAE=   7.283 RMSE=  13.998 R2= 0.998


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


In [15]:
# ─── Version B: Strict common-row comparison ──────────────────────────────────
# All models evaluated on EXACTLY the same observations (rows with ALL features)

ALL_REQUIRED = LR_FEATURES + [TARGET]  # superset: if LR works, all naive models work too

all_results_B = []

for sname, s in eval_sets.items():
    s_common = s.dropna(subset=ALL_REQUIRED)
    if len(s_common) == 0:
        print(f'{sname}: 0 common rows — SKIP')
        continue

    y_true = s_common[TARGET].values

    for model_name, y_hat in [
        ('Persistence_1h',      s_common['load_lag_1h'].values),
        ('SeasonalNaive_24h',   s_common['load_lag_24h'].values),
        ('SeasonalNaive_168h',  s_common['load_lag_168h'].values),
        ('LinearRegression',    lr_model.predict(scaler.transform(s_common[LR_FEATURES]))),
    ]:
        m = compute_metrics(y_true, y_hat)
        m['model']    = model_name
        m['eval_set'] = sname
        all_results_B.append(m)

results_A = pd.DataFrame(all_results_A)[['model','eval_set','N','MAE','RMSE','NRMSE_mean','R2']]
results_B = pd.DataFrame(all_results_B)[['model','eval_set','N','MAE','RMSE','NRMSE_mean','R2']]

results_A.to_csv(DIR_BASELINES / 'baseline_results_A_max_rows.csv', index=False)
results_B.to_csv(DIR_BASELINES / 'baseline_results_B_common_rows.csv', index=False)
print('\nSaved → 05_baselines/baseline_results_A_max_rows.csv')
print('Saved → 05_baselines/baseline_results_B_common_rows.csv')

print('\n=== VERSION A: Maximum valid rows ===')
print(results_A.to_string(index=False))
print('\n=== VERSION B: Strict common rows ===')
print(results_B.to_string(index=False))


Saved → 05_baselines/baseline_results_A_max_rows.csv
Saved → 05_baselines/baseline_results_B_common_rows.csv

=== VERSION A: Maximum valid rows ===
             model                     eval_set      N       MAE      RMSE  NRMSE_mean       R2
    Persistence_1h                   validation  82657  6.320912 15.449266    0.103487 0.996762
    Persistence_1h temporal_test_seen_buildings  80415  6.236205 14.536607    0.100386 0.996753
    Persistence_1h         unseen_building_test 152088  7.842359 17.970840    0.082502 0.996668
    Persistence_1h             joint_shift_test  22461  7.949109 17.005422    0.075904 0.997131
 SeasonalNaive_24h                   validation  82657 12.557738 29.157208    0.195309 0.988465
 SeasonalNaive_24h temporal_test_seen_buildings  80415 12.320400 28.677964    0.198043 0.987362
 SeasonalNaive_24h         unseen_building_test 152088 14.696153 33.705621    0.154738 0.988278
 SeasonalNaive_24h             joint_shift_test  22461 15.065442 31.910250    0.142

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


In [16]:
# ─── Building-level breakdown ─────────────────────────────────────────────────
print('\n=== Building-level Linear Regression performance ===')
bldg_lr_rows = []
for sname, s in eval_sets.items():
    s_valid = s.dropna(subset=LR_FEATURES + [TARGET])
    if len(s_valid) == 0: continue
    for bld in sorted(s_valid['building_id'].unique()):
        sb = s_valid[s_valid['building_id']==bld]
        X_b = scaler.transform(sb[LR_FEATURES])
        y_b = sb[TARGET].values
        yp_b = lr_model.predict(X_b)
        m = compute_metrics(y_b, yp_b)
        m['eval_set'] = sname
        m['building_id'] = bld
        bldg_lr_rows.append(m)

bldg_lr_df = pd.DataFrame(bldg_lr_rows)[['eval_set','building_id','N','MAE','RMSE','NRMSE_mean','R2']]
bldg_lr_df.to_csv(DIR_BASELINES / 'baseline_lr_building_level.csv', index=False)

print('Building-level LR RMSE summary:')
for sname in eval_sets:
    sub = bldg_lr_df[bldg_lr_df['eval_set']==sname]['RMSE']
    if len(sub) == 0: continue
    print(f'  {sname:40s}: mean={sub.mean():.3f}  median={sub.median():.3f}  std={sub.std():.3f}  n_bldg={len(sub)}')

print('\nSaved → 05_baselines/baseline_lr_building_level.csv')

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_



=== Building-level Linear Regression performance ===


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


Building-level LR RMSE summary:
  validation                              : mean=9.567  median=4.931  std=8.771  n_bldg=32
  temporal_test_seen_buildings            : mean=9.013  median=5.103  std=7.550  n_bldg=32
  unseen_building_test                    : mean=10.807  median=6.383  std=11.121  n_bldg=9
  joint_shift_test                        : mean=10.377  median=8.286  std=9.701  n_bldg=9

Saved → 05_baselines/baseline_lr_building_level.csv


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


## Section 11 — Final Audit Report

In [17]:
# ─── Print final concise report ───────────────────────────────────────────────
print('='*70)
print('FINAL PIPELINE AUDIT REPORT')
print('='*70)

print('\nDATASET')
print('-------')
print(f'Raw rows                  : {len(raw):,}')
print(f'Raw buildings             : {raw["building_id"].nunique()}')
print(f'Final eligible buildings  : {len(study_pop)}')
print(f'Final model-ready rows    : {df["supervised_eligible"].sum():,}')

print('\nBUILDING SPLIT')
print('--------------')
print(f'Train buildings           : {len(TRAIN_BLDGS)}')
print(f'Validation buildings      : {len(VAL_BLDGS)}')
print(f'Unseen test buildings     : {len(TEST_BLDGS)}')

print('\nTEMPORAL SPLIT')
print('--------------')
print(f'Train period              : {TRAIN_START} → {TRAIN_END}')
print(f'Validation period         : {VAL_START} → {VAL_END}')
print(f'Future test period        : {TEST_START} → {ts_max}')

print('\nEXPERIMENTAL SETS')
print('-----------------')
for r in summary_df.itertuples():
    print(f'{r.set:40s}: {r.rows:7,} rows  |  {r.buildings} buildings')

print('\nLEAKAGE CHECK')
print('-------------')
print('ALL CHECKS PASSED ✓' if all_pass else 'SOME CHECKS FAILED ✗ — REVIEW ABOVE')

print('\nBASELINE RESULTS (Version B: Common rows, primary metric = RMSE)')
print('------------------------------------------------------------------')
pivot = results_B.pivot_table(index='model', columns='eval_set', values='RMSE')
print(pivot.round(3).to_string())

print('\nARTIFACT DIRECTORY')
print('------------------')
for f in sorted(ARTIFACTS.rglob('*.csv')):
    rel = f.relative_to(PROJECT_ROOT)
    print(f'  {rel}')

print('\n' + '='*70)
print('Pipeline complete. Run All → fully reproducible from recovered_data.csv')
print('='*70)

FINAL PIPELINE AUDIT REPORT

DATASET
-------
Raw rows                  : 1,048,576
Raw buildings             : 60
Final eligible buildings  : 48
Final model-ready rows    : 817,478

BUILDING SPLIT
--------------
Train buildings           : 32
Validation buildings      : 7
Unseen test buildings     : 9

TEMPORAL SPLIT
--------------
Train period              : 2016-01-08 00:00:00 → 2017-05-28 18:00:00
Validation period         : 2017-05-28 19:00:00 → 2017-09-14 08:00:00
Future test period        : 2017-09-14 09:00:00 → 2017-12-31 23:00:00

EXPERIMENTAL SETS
-----------------
train                                   : 384,259 rows  |  32 buildings
validation                              :  82,657 rows  |  32 buildings
temporal_test_seen_buildings            :  80,415 rows  |  32 buildings
building_validation                     : 100,609 rows  |  7 buildings
unseen_building_test                    : 152,088 rows  |  9 buildings
joint_shift_test                        :  22,461 rows  |  9 